# Indian Sign Language (ISL) Translator
## Exploration, In-Memory ZIP Loading, Model Comparison (YOLO11n vs MobileNetV3 vs EfficientNet-Lite0), and Offline ONNX Export

This notebook demonstrates:
1. **Zero-Disk Extraction**: Reading the ISL dataset directly from `dataset_ISL.zip`.
2. **Exploratory Data Analysis (EDA)**: Inspecting classes (`a`-`z`), class distributions, and sample gestures.
3. **Model Architecture Comparison**: Comparing **MobileNetV3-Small**, **EfficientNet-Lite0**, and **YOLO11n**.
4. **Benchmarking**: Parameter count, model size (MB), latency (ms/frame), and FPS.
5. **Offline Deployment Pipeline**: Exporting to ONNX for 100% offline edge/browser inference.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import zipfile
import io
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torchvision.transforms as transforms

from data.dataset import ISLZipDataset, build_dataloaders
from src.models.factory import create_model, count_parameters, get_model_specs
from src.models.export_onnx import export_model_to_onnx

### 1. In-Memory ZIP Dataset Loading (Zero Extraction to Disk)
We inspect and sample images directly from the archive without unzipping anything to the filesystem.

In [ ]:
ZIP_PATH = '../dataset_ISL.zip' if os.path.exists('../dataset_ISL.zip') else 'dataset_ISL.zip'
print(f"Target dataset ZIP path: {os.path.abspath(ZIP_PATH)}")

dataset = ISLZipDataset(zip_path=ZIP_PATH)
print(f"Total images indexed: {len(dataset)}")
print(f"Alphabet classes ({len(dataset.classes)}): {dataset.classes}")

### 2. Visualize Sample Signs Directly from Memory

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
axes = axes.flatten()

# Sample 18 random images across different classes
import random
sample_indices = random.sample(range(len(dataset)), 18)

for ax, idx in zip(axes, sample_indices):
    img, class_label = dataset.get_sample_image(idx)
    ax.imshow(img)
    ax.set_title(f"Sign: '{class_label.upper()}'", fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle("Sample Indian Sign Language Gestures Loaded Directly from ZIP", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 3. Model Specifications & Architectural Trade-offs
We compare the three target architectures:
- **MobileNetV3-Small**: Ultra-low latency, tailored for CPU mobile/edge devices.
- **EfficientNet-Lite0**: High accuracy per parameter, quantized-friendly.
- **YOLO11n**: Modern nano vision backbone with high representational capacity.

In [ ]:
specs = get_model_specs()
specs_df = pd.DataFrame.from_dict(specs, orient='index')
specs_df[['name', 'developer', 'target_use', 'typical_params', 'fp32_size', 'edge_readiness']]

### 4. Empirical Benchmark: Model Footprint, Latency, & FPS

In [ ]:
import time
import numpy as np

models = {
    "MobileNetV3-Small": create_model("mobilenet_v3_small", num_classes=26, pretrained=False),
    "EfficientNet-Lite0": create_model("efficientnet_lite0", num_classes=26, pretrained=False),
    "YOLO11n": create_model("yolo11n", num_classes=26, pretrained=False)
}

results = []
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dummy_input = torch.randn(1, 3, 224, 224).to(device)

for name, m in models.items():
    m.to(device).eval()
    params, size_mb = count_parameters(m)
    
    # Warmup
    with torch.no_grad():
        for _ in range(10): _ = m(dummy_input)
        
    # Benchmark 50 runs
    timings = []
    with torch.no_grad():
        for _ in range(50):
            t0 = time.perf_counter()
            _ = m(dummy_input)
            timings.append((time.perf_counter() - t0) * 1000.0)
            
    avg_lat = np.mean(timings)
    fps = 1000.0 / avg_lat
    results.append({
        "Model": name,
        "Params (M)": round(params / 1e6, 2),
        "Size FP32 (MB)": round(size_mb, 2),
        "Size INT8 (MB)": round(size_mb / 4, 2),
        "Latency (ms)": round(avg_lat, 2),
        "Throughput (FPS)": round(fps, 1)
    })

benchmark_df = pd.DataFrame(results)
benchmark_df

### 5. Benchmark Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Latency comparison
ax1.bar(benchmark_df['Model'], benchmark_df['Latency (ms)'], color=['#3b82f6', '#10b981', '#f59e0b'])
ax1.set_title("Inference Latency per Frame (ms - Lower is Better)", fontweight='bold')
ax1.set_ylabel("Milliseconds")

# FPS comparison
ax2.bar(benchmark_df['Model'], benchmark_df['Throughput (FPS)'], color=['#3b82f6', '#10b981', '#f59e0b'])
ax2.set_title("Throughput (Frames Per Second - Higher is Better)", fontweight='bold')
ax2.set_ylabel("FPS")

plt.tight_layout()
plt.show()

### 6. Exporting to ONNX for Offline Edge & Browser Inference
Offline deployment enables running sign translation directly inside the client's browser using ONNX Runtime WebAssembly, with **zero server dependencies** and **zero internet required**.

In [ ]:
os.makedirs("../models/onnx", exist_ok=True)
onnx_file = "../models/onnx/mobilenet_v3_small.onnx"

export_model_to_onnx(
    model=models["MobileNetV3-Small"],
    output_path=onnx_file,
    input_shape=(1, 3, 224, 224)
)
print(f"ONNX model ready for offline deployment at: {os.path.abspath(onnx_file)}")